In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_absolute_error
from xgboost import XGBClassifier
from scipy.sparse import hstack

In [ ]:
train = pd.read_csv("/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv")
test = pd.read_csv("/content/arvyax_test_inputs_120.xlsx - Sheet1.csv")

In [ ]:
train.head(10)

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5
5,6,after the forest track i feel peaceful and les...,forest,12,8.0,3,2,morning,mixed,calm_face,vague,calm,3
6,7,Nothing strong came up during the rain session...,rain,20,6.5,2,4,early_morning,calm,neutral_face,conflicted,neutral,1
7,8,"even with the mountain session, my mind kept j...",mountain,12,6.0,3,4,morning,neutral,tense_face,clear,restless,4
8,9,I couldn't really settle into the cafe track; ...,cafe,8,5.5,3,4,early_morning,mixed,neutral_face,vague,restless,4
9,10,The mountain ambience helped me stop drifting ...,mountain,15,7.0,4,2,morning,overwhelmed,calm_face,conflicted,focused,3


In [ ]:
test.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear
2,10003,kinda calm ...,cafe,15,8.5,2,5,evening,calm,happy_face,vague
3,10004,after the session i felt able to think straigh...,ocean,7,7.0,2,3,morning,overwhelmed,none,clear
4,10005,lowkey felt pretty grounded. i had to restart ...,ocean,20,8.5,1,5,afternoon,calm,tired_face,vague


In [ ]:
train.isnull().sum()

,0
id,0
journal_text,0
ambience_type,0
duration_min,0
sleep_hours,7
energy_level,0
stress_level,0
time_of_day,0
previous_day_mood,15
face_emotion_hint,123


In [ ]:
test.isnull().sum()

,0
id,0
journal_text,0
ambience_type,0
duration_min,0
sleep_hours,0
energy_level,0
stress_level,0
time_of_day,0
previous_day_mood,10
face_emotion_hint,19


In [ ]:
train.shape

(1200, 13)

In [ ]:
test.shape


(120, 11)

In [ ]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

train['journal_text'] = train['journal_text'].fillna("").apply(clean)
test['journal_text'] = test['journal_text'].fillna("").apply(clean)

In [ ]:
for df in [train, test]:
    df['text_len'] = df['journal_text'].apply(len)
    df['word_count'] = df['journal_text'].apply(lambda x: len(x.split()))

In [ ]:
y_state = train['emotional_state']
y_intensity = train['intensity'] - 1

le = LabelEncoder()
y_state_encoded = le.fit_transform(y_state)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=4000,
    ngram_range=(1,2),
    min_df=2
)
X_text = tfidf.fit_transform(train['journal_text'])
X_text_test = tfidf.transform(test['journal_text'])

In [ ]:
num_cols = ['sleep_hours', 'energy_level', 'stress_level', 'duration_min', 'text_len', 'word_count']

imputer = SimpleImputer(strategy='median')

X_num = imputer.fit_transform(train[num_cols]).astype(float)
X_num_test = imputer.transform(test[num_cols]).astype(float)

In [ ]:
cat_cols = ['ambience_type','time_of_day','previous_day_mood','face_emotion_hint']
X_cat = pd.get_dummies(train[cat_cols], dummy_na=True).astype(float)
cat_columns = X_cat.columns
X_cat_test = pd.get_dummies(test[cat_cols], dummy_na=True).astype(float)
X_cat_test = X_cat_test.reindex(columns=cat_columns, fill_value=0)

In [ ]:
X_final = hstack([X_text, X_cat.values, X_num])
X_test_final = hstack([X_text_test, X_cat_test.values, X_num_test])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

X_train_t, X_val_t, y_train_t, y_val_t = train_test_split(
    X_text, y_state_encoded, test_size=0.2, random_state=42
)

clf_text = XGBClassifier(n_estimators=300)

clf_text.fit(X_train_t, y_train_t)

preds_t = clf_text.predict(X_val_t)

acc_text = accuracy_score(
    le.inverse_transform(y_val_t),
    le.inverse_transform(preds_t)
)
print("TEXT ONLY ACCURACY:", acc_text)

📊 TEXT ONLY ACCURACY: 0.675


In [ ]:
# HYBRID MODEL (TEXT + METADATA)

from scipy.sparse import hstack

X_hybrid = hstack([X_text, X_cat.values, X_num])

X_train_h, X_val_h, y_train_h, y_val_h = train_test_split(
    X_hybrid, y_state_encoded, test_size=0.2, random_state=42
)
clf_hybrid = XGBClassifier(n_estimators=300)

clf_hybrid.fit(X_train_h, y_train_h)

preds_h = clf_hybrid.predict(X_val_h)

acc_hybrid = accuracy_score(
    le.inverse_transform(y_val_h),
    le.inverse_transform(preds_h)
)
print("HYBRID ACCURACY:", acc_hybrid)

HYBRID ACCURACY: 0.6875


In [ ]:
print("\n ABLATION STUDY RESULT")
print(f"Text Only Accuracy     : {acc_text:.4f}")
print(f"Text + Metadata Accuracy: {acc_hybrid:.4f}")

improvement = acc_hybrid - acc_text
print(f"Improvement            : {improvement:.4f}")


 ABLATION STUDY RESULT
Text Only Accuracy     : 0.6750
Text + Metadata Accuracy: 0.6875
Improvement            : 0.0125


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_final, y_state_encoded, test_size=0.2, random_state=42
)
clf = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03
)
clf.fit(X_train, y_train)
preds = clf.predict(X_val)
acc = accuracy_score(le.inverse_transform(y_val), le.inverse_transform(preds))
print("Accuracy:", acc)

Accuracy: 0.6916666666666667


In [ ]:
X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
    X_final,
    y_intensity,
    test_size=0.2,
    random_state=42
)
clf_int = XGBClassifier(n_estimators=300)
clf_int.fit(X_train_i, y_train_i)
pred_i = clf_int.predict(X_val_i)
pred_i = pred_i + 1
y_val_i = y_val_i + 1
mae = mean_absolute_error(y_val_i, pred_i)
print("MAE:", mae)

MAE: 1.5916666666666666


In [ ]:
state_pred = clf.predict(X_test_final)
state_labels = le.inverse_transform(state_pred)

int_pred = clf_int.predict(X_test_final) + 1

In [ ]:
probs = clf.predict_proba(X_test_final)
confidence = np.max(probs, axis=1)
uncertain_flag = (confidence < 0.6).astype(int)

In [ ]:
def decide(state, intensity):
    if intensity >= 4:
        return "deep_work", "now"
    return "rest", "later_today"

def message(state):
    return f"You seem {state}. Take a small step forward."

In [ ]:
def decide(state, intensity, stress, energy, time):
    state = str(state).lower()

    # HIGH INTENSITY
    if intensity >= 4:
        if state in ["overwhelmed", "stressed"]:
            return "box_breathing", "now"
        elif state == "restless":
            return "grounding", "within_15_min"
        elif state == "focused":
            return "deep_work", "now"
        elif state == "calm":
            return "deep_work", "within_15_min"
        elif state == "mixed":
            return "journaling", "now"

    # LOW ENERGY
    if energy <= 2:
        return "rest", "now"

    # MODERATE CASES
    if state == "neutral":
        return "light_planning", "later_today"

    if state == "restless":
        return "movement", "within_15_min"

    if state == "calm":
        return "deep_work", "later_today"

    if state == "focused":
        return "deep_work", "now"

    # NIGHT
    if time == "night":
        return "sound_therapy", "tonight"

    return "light_planning", "later_today"

In [ ]:
import random

def generate_message(state, intensity, action):
    state = str(state).lower()

    high_msgs = [
        f"You seem quite {state}. Let's slow down and try {action}.",
        f"It looks like you're feeling {state}. A {action} could help you reset.",
        f"You're experiencing strong {state} feelings. Try {action} to regain balance."
    ]

    medium_msgs = [
        f"You seem a bit {state}. A small step like {action} might help.",
        f"You're slightly {state}. Consider trying {action}.",
        f"A bit of {state} is noticeable. {action} could improve your state."
    ]

    low_msgs = [
        f"You seem stable. You can continue your current flow.",
        f"You're doing okay. Keep going!",
        f"Everything looks balanced. Maintain your current rhythm."
    ]

    if intensity >= 4:
        return random.choice(high_msgs)
    elif intensity >= 3:
        return random.choice(medium_msgs)
    else:
        return random.choice(low_msgs)

In [ ]:
actions, timings, messages = [], [], []

for i in range(len(test)):
    action, timing = decide(
        state_labels[i],
        int_pred[i],
        test['stress_level'].iloc[i],
        test['energy_level'].iloc[i],
        test['time_of_day'].iloc[i]
    )

    msg = generate_message(state_labels[i], int_pred[i], action)
    actions.append(action)
    timings.append(timing)
    messages.append(msg)

In [ ]:
output = pd.DataFrame({
    "id": test['id'],
    "predicted_state": state_labels,
    "predicted_intensity": int_pred,
    "confidence": confidence,
    "uncertain_flag": uncertain_flag,
    "what_to_do": actions,
    "when_to_do": timings,
    "support_message": messages
})
output.head()

,id,predicted_state,predicted_intensity,confidence,uncertain_flag,what_to_do,when_to_do,support_message
0,10001,focused,4,0.815258,0,deep_work,now,It looks like you're feeling focused. A deep_w...
1,10002,mixed,5,0.628162,0,journaling,now,It looks like you're feeling mixed. A journali...
2,10003,focused,5,0.346773,1,deep_work,now,It looks like you're feeling focused. A deep_w...
3,10004,focused,2,0.746503,0,rest,now,You seem stable. You can continue your current...
4,10005,restless,1,0.223196,1,rest,now,Everything looks balanced. Maintain your curre...
